# PhysicalAI — Colab GPU 백엔드 (ngrok 터널, 자동 연결)

배포된 대시보드(`https://tubular-torte-1df9ea.netlify.app`)의 **Run/Training을 Colab GPU에서 실제 실행**하도록 연결합니다.

- `configs/env.yaml`이 `mock_mode: true`라 Isaac Sim 없이 **numpy mock 환경 + torch(CUDA)** 로 IL/RL 학습이 GPU에서 돌아갑니다.
- 백엔드(`uvicorn api.main:app`)를 띄우고 **ngrok 터널**로 공개 HTTPS URL을 만든 뒤, 그 URL을 **대시보드 레지스트리(Railway)에 자동 등록**합니다.
- 대시보드 **Resources → COLAB GPU**의 '자동 연결'이 켜져 있으면, 등록된 URL을 감지해 **복붙 없이 자동으로 연결**됩니다.

**실행 전 필수:**
1. 런타임 → 런타임 유형 변경 → **T4 GPU**(또는 L4) 선택
2. 무료 ngrok authtoken 준비: <https://dashboard.ngrok.com/get-started/your-authtoken>
3. 위 메뉴 **런타임 → 모두 실행(Run all)** 한 번이면 끝납니다.

> ⚠️ **보안**: 이 노트북을 GitHub에 저장("Save a copy to GitHub")하면 입력한 토큰이 그대로 커밋됩니다. 저장 전에 `NGROK_AUTHTOKEN`을 비우고, 노출 시 즉시 [재발급](https://dashboard.ngrok.com/get-started/your-authtoken)하세요.
> Colab 세션/터널 URL은 임시입니다(수 시간 후 만료, 재시작 시 변경).

## Step 1 — GPU 확인

In [ ]:
import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else "❌ nvidia-smi 실패 — 런타임 유형을 T4 GPU로 변경하세요!")

## Step 2 — 레포 클론 + 의존성 설치

레포의 `demos/ checkpoints/ outputs/`는 Windows 전용 심볼릭 링크라 Colab에선 깨져 있습니다. 제거하고 실제 디렉터리로 다시 만듭니다 (비워둔 상태 → COLLECT가 `episode_0000`부터 생성).

In [ ]:
import subprocess, sys, os, pathlib

REPO = "/content/PhysicalAI"
if not os.path.isdir(REPO):
    print("[1/3] Cloning repo...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/00saridon/PhysicalAI.git", REPO], check=True)
os.chdir(REPO)

for d in ["demos", "checkpoints", "outputs"]:
    p = pathlib.Path(d)
    if p.is_symlink() or p.exists():
        subprocess.run(["rm", "-rf", d])
for d in ["demos", "checkpoints/il", "checkpoints/rl", "outputs/policy", "outputs/dataset"]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print("  dirs ready:", os.listdir(".")[:8], "...")

print("[2/3] Installing deps (torch is already CUDA-ready on Colab)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-api.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)

print("[3/3] Verifying imports...")
import torch
print("  torch", torch.__version__, "| cuda", torch.cuda.is_available())
import importlib.util
print("  stable_baselines3:", importlib.util.find_spec("stable_baselines3") is not None)
print("✅ Step 2 complete")

## Step 3 — 백엔드 + ngrok 터널 기동 + 대시보드 자동 등록

아래 `NGROK_AUTHTOKEN`에 본인 토큰을 붙여넣고 실행하세요. URL이 만들어지면 대시보드 레지스트리(Railway)에 자동 등록되어, **Resources → COLAB GPU**에서 자동 연결됩니다.

In [ ]:
NGROK_AUTHTOKEN = ""  # ← ngrok authtoken 붙여넣기 (GitHub 저장 전 반드시 다시 비우세요)

import subprocess, sys, os, time, json, urllib.request, threading

DASHBOARD = "https://tubular-torte-1df9ea.netlify.app"
DASHBOARD_API = "https://physicalai-production.up.railway.app"  # 자동연결 레지스트리(Railway)
assert NGROK_AUTHTOKEN.strip(), "NGROK_AUTHTOKEN을 입력하세요 — https://dashboard.ngrok.com/get-started/your-authtoken"
os.chdir("/content/PhysicalAI")

# ── 1) 백엔드 기동 (MOCK으로 시작 — 대시보드에서 REAL로 토글) ──
env = {**os.environ, "MOCK_PIPELINE": "true"}
backend = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/PhysicalAI", env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
threading.Thread(target=lambda: [None for _ in backend.stdout], daemon=True).start()  # drain

ok = False
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:8000/api/health", timeout=2); ok = True; break
    except Exception:
        time.sleep(1)
print("✅ backend up" if ok else "❌ backend failed")

# ── 2) ngrok 터널 (SSE를 버퍼 없이 통과) ──
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
ngrok.kill()
tunnel = ngrok.connect(8000, "http")
url = tunnel.public_url.replace("http://", "https://")

# ── 3) 대시보드 레지스트리에 자동 등록 (복붙 없이 자동 연결) ──
try:
    req = urllib.request.Request(DASHBOARD_API + "/api/colab/register",
        data=json.dumps({"url": url}).encode(), method="POST",
        headers={"Content-Type": "application/json"})
    urllib.request.urlopen(req, timeout=15)
    print("✅ 대시보드에 자동 등록됨 — Resources → COLAB GPU에서 자동 연결됩니다")
except Exception as e:
    print("⚠️ 자동 등록 실패 (아래 URL로 수동 연결하세요):", e)

print("\n" + "=" * 64)
print("TUNNEL URL :", url)
print("\n👇 자동 연결이 안 되면 이 주소로 직접 여세요:")
print(f"   {DASHBOARD}/?api={url}")
print("=" * 64)

## Step 4 — 사용 방법

1. **Resources → COLAB GPU** 탭을 열어두면 (자동 연결 ON) 위 등록 직후 대시보드가 자동으로 이 GPU에 붙습니다. (안 되면 출력된 `?api=` 링크로 접속.)
2. 상단 토글을 **REAL_MODE**로 전환 (이 백엔드는 torch가 있어 활성화됨).
3. **Run** 메뉴에서 순서대로 실행: `ENV → COLLECT → IL → RL → EXPORT` (REAL은 COLLECT부터).
4. **Training** / **Resources**에서 reward·loss·GPU 사용률을 실시간으로 확인합니다.

이 노트북 탭은 **열어둔 채로** 두세요 — 닫으면 백엔드와 터널이 종료됩니다.

### 중지

In [ ]:
try:
    from pyngrok import ngrok
    ngrok.kill()
    backend.terminate()
    print("stopped backend + ngrok")
except NameError:
    print("nothing to stop (Step 3 을 먼저 실행하세요)")